***

## Implement co-teaching

In this notebook, we will do the following:
* Load the Assignment data
* Create the base model
* Train it
* Get the loss values

***

In [1]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 26.1 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np

# from google.colab import drive
import seaborn as sns

from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms
from torchvision import datasets, models
import optuna

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torchvision


In [3]:
sns.set_style(style="darkgrid")

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Your original drive path
drive_path = "/content/drive/MyDrive/project_CNN_noisy_label/Dataset 1"
# New local path
local_path = "/content/local_dataset"

# Copy the training data locally
shutil.copytree(f"{drive_path}/train", f"{local_path}/train")

# Update your path variables
image_dir_train = f"{local_path}/train"

Mounted at /content/drive


In [6]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
import cv2
import matplotlib.pyplot as plt
import os
import shutil # New Import for file operations
from google.colab import drive

drive.mount('/content/drive')

print("CUDA available:", torch.cuda.is_available())

# #############################################
# #            Data Copy (The Fix for Slow I/O)
# #############################################

# # Define original paths on Google Drive
base_dir_drive = "/content/drive/MyDrive/project_CNN_noisy_label/Dataset 1"
# image_dir_train_drive = f"{base_dir_drive}/train"
image_dir_validation_drive = f"{base_dir_drive}/Dataset1"

# # Define new local paths for fast I/O
local_base_dir = "/content/local_dataset"
image_dir_train = f"{local_base_dir}/train"
image_dir_validation = f"{local_base_dir}/validation" # Renaming to 'validation' for clarity

# # Create the local directories
# os.makedirs(image_dir_train, exist_ok=True)
# os.makedirs(image_dir_validation, exist_ok=True)

# print("\n[INFO] Starting data copy from Google Drive to local storage...")
try:
#     # Copy the training data
#     shutil.copytree(image_dir_train_drive, image_dir_train, dirs_exist_ok=True)
#     print(f"[INFO] Copied training data to {image_dir_train}")

#     # Copy the validation data (Note: assuming 'Dataset1' is the validation folder name)
    shutil.copytree(image_dir_validation_drive, image_dir_validation, dirs_exist_ok=True)
    print(f"[INFO] Copied validation data to {image_dir_validation}")

except Exception as e:
    print(f"[ERROR] Failed to copy data. Ensure the paths are correct: {e}")

# #############################################
# #            Visualize Images (Using NEW local path)
# #############################################

# Now using the local path: image_dir_train
class_names = [i for i in os.listdir(image_dir_train)
               if os.path.isdir(os.path.join(image_dir_train, i))]
print("\nClasses:", class_names)

# Visualization loop (omitted for brevity, keep your original visualization code here)

#############################################
#            Transforms
#############################################

train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomResizedCrop(128),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

image_dataset = ImageFolder(image_dir_train, transform=train_transforms)
image_dataset_validation = ImageFolder(image_dir_validation, transform=valid_transform)

class_to_idx = image_dataset.class_to_idx
print("class_to_idx:", class_to_idx)

#############################################
#            Data Loaders
#############################################

# num_workers=4 and pin_memory=True are essential for V6/V100 GPUs
train_dataloader = DataLoader(
    image_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_dataloader = DataLoader(
    image_dataset_validation,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#############################################
# Convert Validation Data to Tensor
#############################################

X_valid_list = []
y_valid_list = []

for images, labels in test_dataloader:
    # Use non_blocking=True for fast transfer if not already
    X_valid_list.append(images)
    y_valid_list.append(labels)

X_valid_tensor = torch.cat(X_valid_list, dim=0).to(device)
y_valid_tensor = torch.cat(y_valid_list, dim=0).to(device)

print("\nShape of X_valid_tensor:", X_valid_tensor.shape)
print("Shape of y_valid_tensor:", y_valid_tensor.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA available: True
[INFO] Copied validation data to /content/local_dataset/validation

Classes: ['n01833805', 'n01882714', 'n01820546', 'n01843383', 'n01872401', 'n01860187', 'n01855672', 'n01843065', 'n02012849', 'n01855032']
class_to_idx: {'n01820546': 0, 'n01833805': 1, 'n01843065': 2, 'n01843383': 3, 'n01855032': 4, 'n01855672': 5, 'n01860187': 6, 'n01872401': 7, 'n01882714': 8, 'n02012849': 9}

Shape of X_valid_tensor: torch.Size([500, 3, 128, 128])
Shape of y_valid_tensor: torch.Size([500])


In [8]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is available. Using:", device)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using:", device)
else:
    device = torch.d4evi4ce("cpu")
    print("MPS not available. Using CPU instead.")

CUDA is available. Using: cuda


In [9]:
save_dir = "/content/drive/MyDrive/project_CNN_noisy_label/Dataset 1/Noisy_label_result"
os.makedirs(save_dir, exist_ok=True)

In [10]:
def call_bn(bn, x):
    return bn(x)

import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_small(nn.Module):
    def __init__(self, input_channel=3, n_outputs=10, dropout_rate=0.25, top_bn=False):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.top_bn = top_bn

        # conv blocks
        self.c1 = nn.Conv2d(input_channel, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.act1 = nn.LeakyReLU(0.01, inplace=True)

        self.c2 = nn.Conv2d(64, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.act2 = nn.LeakyReLU(0.01, inplace=True)

        self.c3 = nn.Conv2d(64, 32, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.act3 = nn.LeakyReLU(0.01, inplace=True)

        self.c4 = nn.Conv2d(32, 32, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(32)
        self.act4 = nn.LeakyReLU(0.01, inplace=True)

        self.c5 = nn.Conv2d(32, 32, 3, padding=1)
        self.bn5 = nn.BatchNorm2d(32)
        self.act5 = nn.LeakyReLU(0.01, inplace=True)

        # self.c6 = nn.Conv2d(256, 256, 3, padding=1)
        # self.bn6 = nn.BatchNorm2d(256)
        # self.act6 = nn.LeakyReLU(0.01, inplace=True)

        # self.c7 = nn.Conv2d(256, 512, 3, padding=0)
        # self.bn7 = nn.BatchNorm2d(512)
        # self.act7 = nn.LeakyReLU(0.01, inplace=True)

        # self.c8 = nn.Conv2d(512, 256, 3, padding=0)
        # self.bn8 = nn.BatchNorm2d(256)
        # self.act8 = nn.LeakyReLU(0.01, inplace=True)

        # self.c9 = nn.Conv2d(256, 128, 3, padding=0)
        # self.bn9 = nn.BatchNorm2d(128)
        # self.act9 = nn.LeakyReLU(0.01, inplace=True)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, n_outputs)

    def forward(self, x):
        x = self.act1(self.bn1(self.c1(x)))
        x = self.act2(self.bn2(self.c2(x)))
        x = self.act3(self.bn3(self.c3(x)))
        x = F.max_pool2d(x, 2, 2)
        x = F.dropout2d(x, p=self.dropout_rate, training=self.training)

        x = self.act4(self.bn4(self.c4(x)))
        x = self.act5(self.bn5(self.c5(x)))
        # x = self.act6(self.bn6(self.c6(x)))
        # x = F.max_pool2d(x, 2, 2)
        # x = F.dropout2d(x, p=self.dropout_rate, training=self.training)

        # x = self.act7(self.bn7(self.c7(x)))
        # x = self.act8(self.bn8(self.c8(x)))
        # x = self.act9(self.bn9(self.c9(x)))

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np

LEARNING_RATE = 0.01
NUM_EPOCHS=30


# Initialize the model (using CNN_small from the first snippet)
Base_Model = CNN_small().to(device)

# Define Loss and Optimizer (standard pattern from the second snippet, but with LR from the first snippet)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(Base_Model.parameters(), lr=LEARNING_RATE)
# Note: A learning rate of 0.01 is often high for deep CNNs with Adam,
# but we use it as requested by the first snippet's variables.

print(f"--- Starting Training on {device} ---")
print(f"Model: CNN_small")
print(f"Epochs: {NUM_EPOCHS}, Learning Rate: {LEARNING_RATE}")
print(f"Total training batches: {len(train_dataloader)}")

# --- Training Loop (Pattern from the second snippet) ---

train_loss = []
train_accuracy=[]
test_accuracy = []

for epoch in range(NUM_EPOCHS):
    Base_Model.train()
    running_loss = 0.0

    # Training Phase
    for batch_idx, (images, labels) in enumerate(train_dataloader):
        images, labels = images.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = Base_Model(images)

        # Calculate loss (Standard CrossEntropyLoss, replacing co-teaching loss)
        loss = criterion(outputs, labels)

        # Backward pass and optimization step
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Print batch progress every 100 batches
        if batch_idx % 100 == 0:
            print(f"Epoch: {epoch+1:02d}/{NUM_EPOCHS:02d} | Batch: {batch_idx:03d}/{len(train_dataloader):03d} | Loss: {loss.item():.4f}")


    # Calculate average training loss for the epoch
    avg_train_loss = running_loss / len(train_dataloader)
    train_loss.append(avg_train_loss)

    # Calculate training accuracy
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = Base_Model(images)
            predicted = outputs.argmax(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    train_accuracy_score = 100 * correct / total
    train_accuracy.append(train_accuracy_score)
    # Evaluation Phase
    Base_Model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = Base_Model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracy_score = 100 * correct / total
    test_accuracy.append(test_accuracy_score)

    # Print Epoch Summary (Pattern from the second snippet)
    print(f"\n--- Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} Summary ---")
    print(f"Train Loss = {avg_train_loss:.4f}")
    print(f"Train Accuracy = {train_accuracy_score:.2f}%")
    print(f"Test Accuracy = {test_accuracy_score:.2f}%\n")

print("--- Training Complete ---")

--- Starting Training on cuda ---
Model: CNN_small
Epochs: 30, Learning Rate: 0.01
Total training batches: 193
Epoch: 01/30 | Batch: 000/193 | Loss: 2.3190
Epoch: 01/30 | Batch: 100/193 | Loss: 2.0393

--- Epoch 01/30 Summary ---
Train Loss = 1.9742
Train Accuracy = 34.91%
Test Accuracy = 38.20%

Epoch: 02/30 | Batch: 000/193 | Loss: 1.8241
Epoch: 02/30 | Batch: 100/193 | Loss: 1.7898

--- Epoch 02/30 Summary ---
Train Loss = 1.8261
Train Accuracy = 37.55%
Test Accuracy = 39.60%

Epoch: 03/30 | Batch: 000/193 | Loss: 1.9663
Epoch: 03/30 | Batch: 100/193 | Loss: 1.9179

--- Epoch 03/30 Summary ---
Train Loss = 1.7676
Train Accuracy = 39.39%
Test Accuracy = 41.40%

Epoch: 04/30 | Batch: 000/193 | Loss: 1.7269
Epoch: 04/30 | Batch: 100/193 | Loss: 1.7298

--- Epoch 04/30 Summary ---
Train Loss = 1.7183
Train Accuracy = 42.90%
Test Accuracy = 44.00%

Epoch: 05/30 | Batch: 000/193 | Loss: 1.6325
Epoch: 05/30 | Batch: 100/193 | Loss: 1.4788

--- Epoch 05/30 Summary ---
Train Loss = 1.6666
Tr

In [12]:
import os

save_dir = "/content/drive/MyDrive/project_CNN_noisy_label/Dataset 1/Noisy_label_result"

os.makedirs(save_dir, exist_ok=True)

print("Folder created at:", save_dir)

print(train_accuracy)
print(test_accuracy)
print(train_loss)
torch.save(Base_Model,f"{save_dir}/Base_Model_Assignment.pth")
metrics = {
    "Name":"BaseModel CNN Assignment",
    "train_loss": train_loss,
    "train_accuracy": train_accuracy,
    "test_accuracy": test_accuracy
}

torch.save(metrics,
           f"{save_dir}/training_metrics_BASEMODEL_Assignment.pth")

Folder created at: /content/drive/MyDrive/project_CNN_noisy_label/Dataset 1/Noisy_label_result
[34.91330416464106, 37.54658888348728, 39.39393939393939, 42.902284880894506, 41.370928536703936, 44.798249878463785, 47.26948630691946, 47.131745260087506, 47.98249878463782, 49.927078269324255, 51.01280181494085, 52.69810403500243, 53.532652730513696, 54.51304488737644, 54.829039053637985, 55.83373845405931, 56.9194619996759, 58.3697941986712, 58.51563766002268, 60.13612056392805, 58.66958353589369, 59.55274671852212, 61.570247933884296, 60.60606060606061, 62.218441095446444, 61.270458596661804, 62.980068060281965, 64.61675579322637, 62.34807972775887, 64.12250850753524]
[38.2, 39.6, 41.4, 44.0, 42.4, 46.4, 48.0, 46.6, 51.2, 52.2, 56.4, 55.4, 54.4, 59.8, 59.4, 58.4, 60.6, 62.0, 60.8, 61.8, 65.2, 64.6, 66.2, 64.6, 63.4, 66.0, 65.8, 63.8, 68.4, 70.0]
[1.974245543924638, 1.8261451288826107, 1.7675708754692672, 1.718327696459281, 1.6666493181119928, 1.6406811185451369, 1.6035658683183898, 1.572

In [13]:
metrics

{'Name': 'BaseModel CNN Assignment',
 'train_loss': [1.974245543924638,
  1.8261451288826107,
  1.7675708754692672,
  1.718327696459281,
  1.6666493181119928,
  1.6406811185451369,
  1.6035658683183898,
  1.5728839339369938,
  1.546692726525618,
  1.501334997038767,
  1.4707178501267508,
  1.4291821860278826,
  1.4018529665902488,
  1.3676590567425744,
  1.3352528087833384,
  1.320096211112225,
  1.2919378292992942,
  1.2437334008167444,
  1.251885438212459,
  1.206285056981398,
  1.1934032257974456,
  1.1895508065124867,
  1.188864600164285,
  1.1650446353798702,
  1.1435282693625732,
  1.1406730706827628,
  1.1143879127625975,
  1.1077063966909222,
  1.0727484494910957,
  1.0802132814659355],
 'train_accuracy': [34.91330416464106,
  37.54658888348728,
  39.39393939393939,
  42.902284880894506,
  41.370928536703936,
  44.798249878463785,
  47.26948630691946,
  47.131745260087506,
  47.98249878463782,
  49.927078269324255,
  51.01280181494085,
  52.69810403500243,
  53.532652730513696,

In [ ]:
444